# Transfer Learning Localization Model - 5-Fold Cross Validation

Notebook ini memakai data split yang sudah ada di `data_ready`: `train` dan `val` digabung menjadi dataset cross-validation, sedangkan `test` tetap menjadi holdout final pada setiap fold. Tidak ada lagi pengkondisian khusus LMI karena distribusi data sudah ditangani sebelumnya melalui GAN.

Arsitektur model disamakan dengan `pretrained_model_localization.ipynb`: `CNNLSTMSEAttentionLocalization` dengan lead-wise attention, SE blocks, BiLSTM, dan temporal attention.


## 1. Import dan Konfigurasi


In [1]:
import os
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_curve,
    auc,
    average_precision_score,
    precision_recall_curve,
)
from sklearn.preprocessing import label_binarize


In [2]:
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
DATA_DIR = PROJECT_ROOT / 'data_ready_ptb'
OUTPUT_DIR = PROJECT_ROOT / 'crossval_comparison' / 'localization_architecture_5fold'
PRETRAIN_PATH = PROJECT_ROOT / 'pretrain_model_localization' / 'best_model_localization_evidence.pth'

class_names = ["NORM", "IMI", "AMI", "LMI"]
label_to_idx = {name: i for i, name in enumerate(class_names)}
lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

CONFIG = {
    'seed': 42,
    'n_splits': 5,
    'batch_size': 32,
    'epochs': 100,
    'use_early_stopping': False,
    'patience': None,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'gradient_clip_norm': 2.0,
    'focal_gamma': 1.5,
    'use_weighted_sampler': True,
    'selection_metric': 'macro_f1',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

MODEL_CONFIGS = [
    {
        'model_name': 'localization_no_pretrain',
        'pretrain_path': None,
        'freeze_mode': 'full',
        'learning_rate': 1e-3,
    },
    {
        'model_name': 'localization_frozen_backbone',
        'pretrain_path': PRETRAIN_PATH,
        'freeze_mode': 'frozen_backbone',
        'learning_rate': 1e-3,
    },
    {
        'model_name': 'localization_partial_finetune',
        'pretrain_path': PRETRAIN_PATH,
        'freeze_mode': 'partial',
        'learning_rate': 5e-4,
    },
    {
        'model_name': 'localization_full_finetune',
        'pretrain_path': PRETRAIN_PATH,
        'freeze_mode': 'full',
        'learning_rate': 1e-4,
    },
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
device = torch.device(CONFIG['device'])

print(json.dumps(CONFIG, indent=2))
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('PRETRAIN_PATH:', PRETRAIN_PATH)
print('Device:', device)


{
  "seed": 42,
  "n_splits": 5,
  "batch_size": 32,
  "epochs": 100,
  "use_early_stopping": false,
  "patience": null,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "gradient_clip_norm": 2.0,
  "focal_gamma": 1.5,
  "use_weighted_sampler": true,
  "selection_metric": "macro_f1",
  "device": "cuda"
}
DATA_DIR: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/data_ready_ptb
OUTPUT_DIR: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/crossval_comparison/localization_architecture_5fold
PRETRAIN_PATH: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/pretrain_model_localization/best_model_localization_evidence.pth
Device: cuda


## 2. Load Data: Train + Val Digabung


In [3]:
def load_split(data_dir, split):
    x = np.load(data_dir / f'x_{split}.npy').astype(np.float32)
    y = np.load(data_dir / f'y_{split}.npy')
    if y.ndim > 1 and y.shape[1] > 1:
        y_idx = y.argmax(axis=1).astype(np.int64)
    else:
        y_idx = y.reshape(-1).astype(np.int64)
    return x, y_idx

def validate_ecg_array(x, name):
    if x.ndim != 3:
        raise ValueError(f'{name}: expected [N, length, leads], got {x.shape}')
    if x.shape[2] != len(lead_names):
        raise ValueError(f'{name}: expected 12 leads, got {x.shape[2]}')
    if not np.isfinite(x).all():
        raise ValueError(f'{name}: contains NaN or Inf')

x_train, y_train = load_split(DATA_DIR, 'train')
x_val, y_val = load_split(DATA_DIR, 'val')
x_test, y_test = load_split(DATA_DIR, 'test')

for split_name, x in [('train', x_train), ('val', x_val), ('test', x_test)]:
    validate_ecg_array(x, split_name)

# Train dan validation lama digabung untuk 5-fold CV. Test tetap holdout final.
x_cv = np.concatenate([x_train, x_val], axis=0).astype(np.float32)
y_cv = np.concatenate([y_train, y_val], axis=0).astype(np.int64)
x_test = x_test.astype(np.float32)
y_test = y_test.astype(np.int64)

print('Combined CV data:', x_cv.shape, np.bincount(y_cv, minlength=len(class_names)))
print('Holdout test    :', x_test.shape, np.bincount(y_test, minlength=len(class_names)))
print('\nClass distribution:')
for idx, name in enumerate(class_names):
    print(f'{name:4} | CV: {int((y_cv == idx).sum()):4d} | Test: {int((y_test == idx).sum()):4d}')


Combined CV data: (163, 1000, 12) [64 64 33  2]
Holdout test    : (42, 1000, 12) [16 16  9  1]

Class distribution:
NORM | CV:   64 | Test:   16
IMI  | CV:   64 | Test:   16
AMI  | CV:   33 | Test:    9
LMI  | CV:    2 | Test:    1


## 3. Dataset, Normalisasi Per Fold, dan Arsitektur Localization


In [4]:
class PerLeadZScore:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        self.mean_ = x.mean(axis=(0, 1), keepdims=True)
        self.std_ = np.maximum(x.std(axis=(0, 1), keepdims=True), self.eps)
        return self

    def transform(self, x):
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError('Normalizer must be fit on the current train fold first.')
        return ((x - self.mean_) / self.std_).astype(np.float32)

class ECGDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

class SE1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Conv1d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv1d(hidden, channels, kernel_size=1)

    def forward(self, x):
        w = self.gap(x)
        w = F.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class LeadWiseAttention(nn.Module):
    def __init__(self, n_leads=12, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        # x: [B, L, C]. Feature per lead: mean, std, mean absolute amplitude.
        mean = x.mean(dim=1)
        std = x.std(dim=1)
        abs_mean = x.abs().mean(dim=1)
        lead_features = torch.stack([mean, std, abs_mean], dim=-1)  # [B, C, 3]
        logits = self.net(lead_features).squeeze(-1)  # [B, C]
        weights = F.softmax(logits, dim=1)
        x_weighted = x * weights.unsqueeze(1)
        return x_weighted, weights

class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(dim, dim) * 0.02)
        self.u = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, x):
        # x: [B, T, D]
        scores = torch.tanh(x @ self.W) @ self.u
        weights = F.softmax(scores, dim=1)
        context = (x * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights

class CNNLSTMSEAttentionLocalization(nn.Module):
    def __init__(self, in_leads=12, num_classes=4, dropout=0.35):
        super().__init__()
        self.in_leads = in_leads
        self.lead_attention = LeadWiseAttention(n_leads=in_leads)

        self.conv1 = nn.Conv1d(in_leads, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.se1 = SE1D(64)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.se2 = SE1D(128)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.se3 = SE1D(128)
        self.pool3 = nn.MaxPool1d(2)

        self.conv4 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm1d(256)
        self.se4 = SE1D(256)

        self.lstm1 = nn.LSTM(input_size=256, hidden_size=128, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(input_size=256, hidden_size=96, batch_first=True, bidirectional=True)
        self.temporal_attention = TemporalAttention(192)
        self.timedense = nn.Linear(192, 128)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, num_classes)

    def forward_features(self, x):
        # Accept [B, L, C].
        x_weighted, lead_weights = self.lead_attention(x)
        x_c = x_weighted.permute(0, 2, 1)  # [B, C, L]

        x_c = self.pool1(self.se1(F.relu(self.bn1(self.conv1(x_c)))))
        x_c = self.pool2(self.se2(F.relu(self.bn2(self.conv2(x_c)))))
        x_c = self.pool3(self.se3(F.relu(self.bn3(self.conv3(x_c)))))
        x_c = self.se4(F.relu(self.bn4(self.conv4(x_c))))

        x_t = x_c.permute(0, 2, 1)
        x_t, _ = self.lstm1(x_t)
        x_t, _ = self.lstm2(x_t)
        context, temporal_weights = self.temporal_attention(x_t)
        features = self.dropout(F.relu(self.timedense(context)))
        return features, lead_weights, temporal_weights

    def forward(self, x, return_attention=False):
        features, lead_weights, temporal_weights = self.forward_features(x)
        logits = self.classifier(features)
        if return_attention:
            return logits, {'lead_attention': lead_weights, 'temporal_attention': temporal_weights}
        return logits

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none', weight=self.weight)
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


## 4. Helper Training, Transfer Learning, dan Plot


In [5]:
def make_loaders_for_fold(train_idx, val_idx, batch_size):
    normalizer = PerLeadZScore().fit(x_cv[train_idx])
    x_fold_train = normalizer.transform(x_cv[train_idx])
    x_fold_val = normalizer.transform(x_cv[val_idx])
    x_fold_test = normalizer.transform(x_test)

    y_fold_train = y_cv[train_idx]
    y_fold_val = y_cv[val_idx]

    train_ds = ECGDataset(x_fold_train, y_fold_train)
    val_ds = ECGDataset(x_fold_val, y_fold_val)
    test_ds = ECGDataset(x_fold_test, y_test)

    if CONFIG['use_weighted_sampler']:
        counts = np.bincount(y_fold_train, minlength=len(class_names))
        weights = 1.0 / np.maximum(counts, 1)
        sample_weights = weights[y_fold_train]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, normalizer

def load_localization_pretrain(model, pretrain_path):
    if pretrain_path is None:
        return {'loaded': False, 'missing_keys': [], 'unexpected_keys': [], 'load_mode': 'no_pretrain'}

    pretrain_path = Path(pretrain_path)
    if not pretrain_path.exists():
        warnings.warn(f'Pretrain checkpoint not found: {pretrain_path}. Model starts from random initialization.', RuntimeWarning)
        return {'loaded': False, 'missing_keys': [], 'unexpected_keys': [], 'load_mode': 'missing'}

    try:
        checkpoint = torch.load(pretrain_path, map_location='cpu', weights_only=True)
        load_mode = 'weights_only=True'
    except Exception as exc:
        # PyTorch 2.6 defaults to weights_only=True. The local localization checkpoint
        # stores numpy metadata, so a trusted local checkpoint may need full pickle load.
        warnings.warn(
            f'weights_only=True failed for local checkpoint {pretrain_path.name}: {exc}. '
            'Retrying with weights_only=False because this checkpoint is produced by this project.',
            RuntimeWarning,
        )
        checkpoint = torch.load(pretrain_path, map_location='cpu', weights_only=False)
        load_mode = 'weights_only=False'

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    result = model.load_state_dict(state_dict, strict=False)
    return {
        'loaded': True,
        'missing_keys': list(result.missing_keys),
        'unexpected_keys': list(result.unexpected_keys),
        'load_mode': load_mode,
    }

def configure_trainable_layers(model, freeze_mode):
    for p in model.parameters():
        p.requires_grad = True

    if freeze_mode == 'full':
        return

    if freeze_mode == 'frozen_backbone':
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith('classifier')
        return

    if freeze_mode == 'partial':
        trainable_keywords = ('conv4', 'bn4', 'se4', 'lstm2', 'temporal_attention', 'timedense', 'classifier')
        for name, p in model.named_parameters():
            p.requires_grad = any(key in name for key in trainable_keywords)
        return

    raise ValueError(f'Unknown freeze_mode: {freeze_mode}')

def count_trainable_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_count += xb.size(0)
    return total_loss / max(total_count, 1), total_correct / max(total_count, 1)



def safe_balanced_accuracy(y_true, y_pred, labels=None):
    if labels is None:
        labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    support = cm.sum(axis=1)
    present = support > 0
    if not np.any(present):
        return np.nan
    recalls = np.divide(
        np.diag(cm),
        support,
        out=np.zeros_like(support, dtype=float),
        where=support > 0,
    )
    return float(recalls[present].mean())

def evaluate_model(model, loader, criterion=None):
    model.eval()
    total_loss, total_count = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            if criterion is not None:
                loss = criterion(logits, yb)
                total_loss += loss.item() * xb.size(0)
                total_count += xb.size(0)
            prob = torch.softmax(logits, dim=1)
            y_true.append(yb.cpu().numpy())
            y_pred.append(prob.argmax(dim=1).cpu().numpy())
            y_prob.append(prob.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_prob = np.vstack(y_prob)
    summary = {
        'loss': total_loss / max(total_count, 1) if criterion is not None else np.nan,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': safe_balanced_accuracy(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    return summary, y_true, y_pred, y_prob

def plot_training_curve(history, title, save_path):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    axes[0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0].plot(df['epoch'], df['val_loss'], label='Val', linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(df['epoch'], df['train_acc'], label='Train')
    axes[1].plot(df['epoch'], df['val_acc'], label='Val', linestyle='--')
    axes[1].plot(df['epoch'], df['val_macro_f1'], label='Val Macro F1', linestyle=':')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(7, 6), dpi=150)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return cm

def plot_roc_pr_curves(y_true, y_prob, title_prefix, save_path):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        axes[0].plot(fpr, tpr, label=f'{name} AUC={roc_auc:.3f}')
        axes[1].plot(recall, precision, label=f'{name} AP={ap:.3f}')

    axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    axes[0].set_title(f'{title_prefix} ROC')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    axes[1].set_title(f'{title_prefix} Precision-Recall')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def classification_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        labels=list(range(len(class_names))),
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).T


## 5. Jalankan 5-Fold Cross Validation


In [6]:
skf = StratifiedKFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=CONFIG['seed'])
all_fold_summaries = []
all_histories = {}

for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg['model_name']
    model_dir = OUTPUT_DIR / model_name
    (model_dir / 'fold_models').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_histories').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_plots').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_reports').mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 90}\nMODEL: {model_name}\n{'=' * 90}")

    for fold, (train_idx, val_idx) in enumerate(skf.split(x_cv, y_cv), start=1):
        set_seed(CONFIG['seed'] + fold)
        fold_dir = model_dir / 'fold_plots' / f'fold{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)

        train_loader, val_loader, test_loader, normalizer = make_loaders_for_fold(
            train_idx, val_idx, CONFIG['batch_size']
        )

        class_counts = np.bincount(y_cv[train_idx], minlength=len(class_names))
        class_weights_np = class_counts.sum() / np.maximum(class_counts, 1)
        class_weights_np = class_weights_np / class_weights_np.mean()
        class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

        model = CNNLSTMSEAttentionLocalization(in_leads=len(lead_names), num_classes=len(class_names)).to(device)
        load_info = load_localization_pretrain(model, model_cfg['pretrain_path'])
        configure_trainable_layers(model, model_cfg['freeze_mode'])
        total_params, trainable_params = count_trainable_parameters(model)

        criterion = FocalLoss(gamma=CONFIG['focal_gamma'], weight=class_weights)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=model_cfg.get('learning_rate', CONFIG['learning_rate']),
            weight_decay=CONFIG['weight_decay'],
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

        best_score = -np.inf
        best_epoch = 0
        best_state = None
        epochs_without_improvement = 0
        history = []

        print(f"\nFold {fold}/{CONFIG['n_splits']} | train={len(train_idx)} val={len(val_idx)} | pretrained={load_info['loaded']} | trainable={trainable_params:,}/{total_params:,}")

        for epoch in range(1, CONFIG['epochs'] + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
            val_summary, _, _, _ = evaluate_model(model, val_loader, criterion)
            selection_score = val_summary[CONFIG['selection_metric']]
            scheduler.step(selection_score)

            row = {
                'model_name': model_name,
                'fold': fold,
                'epoch': epoch,
                'train_loss': train_loss,
                'train_acc': train_acc,
                'val_loss': val_summary['loss'],
                'val_acc': val_summary['accuracy'],
                'val_balanced_accuracy': val_summary['balanced_accuracy'],
                'val_macro_f1': val_summary['macro_f1'],
                'val_weighted_f1': val_summary['weighted_f1'],
                'learning_rate': optimizer.param_groups[0]['lr'],
            }
            history.append(row)

            if selection_score > best_score:
                best_score = selection_score
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG['epochs']:
                print(
                    f"Epoch {epoch:03d} | train_loss={train_loss:.4f} val_loss={val_summary['loss']:.4f} "
                    f"val_acc={val_summary['accuracy']:.4f} val_macro_f1={val_summary['macro_f1']:.4f}"
                )

            if CONFIG.get('use_early_stopping', False) and epochs_without_improvement >= CONFIG['patience']:
                print(f"Early stopping at epoch {epoch}; best epoch {best_epoch} ({CONFIG['selection_metric']}={best_score:.4f})")
                break

        model.load_state_dict(best_state)
        hist_df = pd.DataFrame(history)
        hist_path = model_dir / 'fold_histories' / f'fold{fold}_history.csv'
        hist_df.to_csv(hist_path, index=False)
        all_histories[f'{model_name}_fold{fold}'] = hist_df

        ckpt_path = model_dir / 'fold_models' / f'fold{fold}_best_epoch{best_epoch}_{CONFIG["selection_metric"]}{best_score:.4f}.pth'
        torch.save({
            'model_state_dict': best_state,
            'model_name': model_name,
            'fold': fold,
            'best_epoch': best_epoch,
            'best_score': best_score,
            'class_names': class_names,
            'lead_names': lead_names,
            'normalizer_mean': normalizer.mean_,
            'normalizer_std': normalizer.std_,
            'config': CONFIG,
            'model_config': model_cfg,
            'pretrain_load_info': load_info,
        }, ckpt_path)

        plot_training_curve(hist_df, f'{model_name} - Fold {fold}', fold_dir / 'training_curve.png')

        val_summary, y_val_true, y_val_pred, y_val_prob = evaluate_model(model, val_loader, criterion)
        test_summary, y_test_true, y_test_pred, y_test_prob = evaluate_model(model, test_loader, criterion)

        val_cm = plot_confusion_matrix(
            y_val_true, y_val_pred,
            f'{model_name} Fold {fold} - Validation Confusion Matrix',
            fold_dir / 'confusion_matrix_validation.png',
        )
        test_cm = plot_confusion_matrix(
            y_test_true, y_test_pred,
            f'{model_name} Fold {fold} - Test Confusion Matrix',
            fold_dir / 'confusion_matrix_test.png',
        )
        plot_roc_pr_curves(y_val_true, y_val_prob, f'{model_name} Fold {fold} Validation', fold_dir / 'roc_pr_validation.png')
        plot_roc_pr_curves(y_test_true, y_test_prob, f'{model_name} Fold {fold} Test', fold_dir / 'roc_pr_test.png')

        classification_report_df(y_val_true, y_val_pred).to_csv(model_dir / 'fold_reports' / f'fold{fold}_validation_classification_report.csv')
        classification_report_df(y_test_true, y_test_pred).to_csv(model_dir / 'fold_reports' / f'fold{fold}_test_classification_report.csv')
        pd.DataFrame(val_cm, index=class_names, columns=class_names).to_csv(model_dir / 'fold_reports' / f'fold{fold}_validation_confusion_matrix.csv')
        pd.DataFrame(test_cm, index=class_names, columns=class_names).to_csv(model_dir / 'fold_reports' / f'fold{fold}_test_confusion_matrix.csv')

        summary_row = {
            'model_name': model_name,
            'fold': fold,
            'best_epoch': best_epoch,
            'best_selection_metric': CONFIG['selection_metric'],
            'best_selection_score': best_score,
            'pretrained_loaded': load_info['loaded'],
            'freeze_mode': model_cfg['freeze_mode'],
            'train_samples': len(train_idx),
            'val_samples': len(val_idx),
            'test_samples': len(y_test),
            'trainable_params': trainable_params,
            'total_params': total_params,
            'checkpoint_path': str(ckpt_path),
            'val_accuracy': val_summary['accuracy'],
            'val_balanced_accuracy': val_summary['balanced_accuracy'],
            'val_macro_f1': val_summary['macro_f1'],
            'val_weighted_f1': val_summary['weighted_f1'],
            'test_accuracy': test_summary['accuracy'],
            'test_balanced_accuracy': test_summary['balanced_accuracy'],
            'test_macro_f1': test_summary['macro_f1'],
            'test_weighted_f1': test_summary['weighted_f1'],
        }
        all_fold_summaries.append(summary_row)
        pd.DataFrame([summary_row]).to_csv(model_dir / 'fold_reports' / f'fold{fold}_summary.csv', index=False)

        print(
            f"Fold {fold} done | val_macro_f1={val_summary['macro_f1']:.4f} "
            f"test_macro_f1={test_summary['macro_f1']:.4f}"
        )

summary_df = pd.DataFrame(all_fold_summaries)
summary_df.to_csv(OUTPUT_DIR / 'summary_per_model_per_fold.csv', index=False)
summary_df.groupby('model_name').agg({
    'val_accuracy': ['mean', 'std'],
    'val_macro_f1': ['mean', 'std'],
    'test_accuracy': ['mean', 'std'],
    'test_macro_f1': ['mean', 'std'],
}).to_csv(OUTPUT_DIR / 'summary_by_model.csv')

with pd.ExcelWriter(OUTPUT_DIR / 'crossval_summary.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='summary_per_fold', index=False)
    summary_df.groupby('model_name').agg({
        'val_accuracy': ['mean', 'std'],
        'val_macro_f1': ['mean', 'std'],
        'test_accuracy': ['mean', 'std'],
        'test_macro_f1': ['mean', 'std'],
    }).to_excel(writer, sheet_name='summary_by_model')
    for sheet_name, hist_df in all_histories.items():
        hist_df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print('Saved summary:', OUTPUT_DIR / 'summary_per_model_per_fold.csv')
print('Saved Excel  :', OUTPUT_DIR / 'crossval_summary.xlsx')
summary_df



MODEL: localization_no_pretrain


/home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(



Fold 1/5 | train=130 val=33 | pretrained=False | trainable=951,341/951,341
Epoch 001 | train_loss=0.8155 val_loss=0.0281 val_acc=0.0000 val_macro_f1=0.0000
Epoch 005 | train_loss=0.0539 val_loss=0.0301 val_acc=0.0000 val_macro_f1=0.0000
Epoch 010 | train_loss=0.0203 val_loss=0.0325 val_acc=0.2121 val_macro_f1=0.1000
Epoch 015 | train_loss=0.0249 val_loss=0.0268 val_acc=0.2121 val_macro_f1=0.1029
Epoch 020 | train_loss=0.0142 val_loss=0.0414 val_acc=0.1515 val_macro_f1=0.0893
Epoch 025 | train_loss=0.0173 val_loss=0.0264 val_acc=0.2121 val_macro_f1=0.1000
Epoch 030 | train_loss=0.0141 val_loss=0.0428 val_acc=0.1818 val_macro_f1=0.1071
Epoch 035 | train_loss=0.0167 val_loss=0.0348 val_acc=0.2121 val_macro_f1=0.1129
Epoch 040 | train_loss=0.0145 val_loss=0.0280 val_acc=0.2121 val_macro_f1=0.1029
Epoch 045 | train_loss=0.0129 val_loss=0.0225 val_acc=0.2121 val_macro_f1=0.0946
Epoch 050 | train_loss=0.0155 val_loss=0.0225 val_acc=0.2121 val_macro_f1=0.0946
Epoch 055 | train_loss=0.0125 val

/home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(
/tmp/ipykernel_77969/1741634970.py:46: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler err

Epoch 010 | train_loss=1.0934 val_loss=0.0315 val_acc=0.4848 val_macro_f1=0.4661
Epoch 015 | train_loss=0.2791 val_loss=0.0682 val_acc=0.3939 val_macro_f1=0.3828
Epoch 020 | train_loss=0.0971 val_loss=0.0719 val_acc=0.3939 val_macro_f1=0.3690
Epoch 025 | train_loss=0.0660 val_loss=0.0846 val_acc=0.3636 val_macro_f1=0.3571
Epoch 030 | train_loss=0.1013 val_loss=0.0818 val_acc=0.3939 val_macro_f1=0.3690
Epoch 035 | train_loss=0.0849 val_loss=0.0816 val_acc=0.3333 val_macro_f1=0.3417
Epoch 040 | train_loss=0.0808 val_loss=0.0875 val_acc=0.3636 val_macro_f1=0.3673
Epoch 045 | train_loss=0.1097 val_loss=0.0912 val_acc=0.3636 val_macro_f1=0.3673
Epoch 050 | train_loss=0.0711 val_loss=0.0878 val_acc=0.3939 val_macro_f1=0.3828
Epoch 055 | train_loss=0.1110 val_loss=0.0763 val_acc=0.4242 val_macro_f1=0.3969
Epoch 060 | train_loss=0.0872 val_loss=0.0755 val_acc=0.3636 val_macro_f1=0.3417
Epoch 065 | train_loss=0.0878 val_loss=0.0933 val_acc=0.3636 val_macro_f1=0.3673
Epoch 070 | train_loss=0.078

...

/tmp/ipykernel_77969/1741634970.py:46: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._recons

Epoch 005 | train_loss=0.0050 val_loss=0.0103 val_acc=0.8125 val_macro_f1=0.8003
Epoch 010 | train_loss=0.0014 val_loss=0.0168 val_acc=0.8125 val_macro_f1=0.8003
Epoch 015 | train_loss=0.0011 val_loss=0.0199 val_acc=0.8750 val_macro_f1=0.8719
Epoch 020 | train_loss=0.0012 val_loss=0.0215 val_acc=0.8750 val_macro_f1=0.8719
Epoch 025 | train_loss=0.0007 val_loss=0.0233 val_acc=0.8750 val_macro_f1=0.8719
Epoch 030 | train_loss=0.0010 val_loss=0.0242 val_acc=0.8750 val_macro_f1=0.6704
Epoch 035 | train_loss=0.0009 val_loss=0.0233 val_acc=0.8750 val_macro_f1=0.8719
Epoch 040 | train_loss=0.0008 val_loss=0.0227 val_acc=0.8750 val_macro_f1=0.8719
Epoch 045 | train_loss=0.0010 val_loss=0.0244 val_acc=0.8750 val_macro_f1=0.8719
Epoch 050 | train_loss=0.0027 val_loss=0.0220 val_acc=0.8750 val_macro_f1=0.8719
Epoch 055 | train_loss=0.0007 val_loss=0.0242 val_acc=0.8750 val_macro_f1=0.8719
Epoch 060 | train_loss=0.0023 val_loss=0.0248 val_acc=0.8750 val_macro_f1=0.8719
Epoch 065 | train_loss=0.001

/tmp/ipykernel_77969/1741634970.py:46: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._recons

Epoch 005 | train_loss=0.0054 val_loss=0.0057 val_acc=0.8438 val_macro_f1=0.8658
Epoch 010 | train_loss=0.0055 val_loss=0.0118 val_acc=0.7812 val_macro_f1=0.7869
Epoch 015 | train_loss=0.2223 val_loss=0.0055 val_acc=0.7500 val_macro_f1=0.7608
Epoch 020 | train_loss=0.0007 val_loss=0.0060 val_acc=0.7812 val_macro_f1=0.7869
Epoch 025 | train_loss=0.0010 val_loss=0.0052 val_acc=0.7812 val_macro_f1=0.7905
Epoch 030 | train_loss=0.0009 val_loss=0.0036 val_acc=0.7812 val_macro_f1=0.7905
Epoch 035 | train_loss=0.0008 val_loss=0.0039 val_acc=0.7812 val_macro_f1=0.7905
Epoch 040 | train_loss=0.0066 val_loss=0.0048 val_acc=0.7812 val_macro_f1=0.7905
Epoch 045 | train_loss=0.0011 val_loss=0.0047 val_acc=0.7812 val_macro_f1=0.7905
Epoch 050 | train_loss=0.0007 val_loss=0.0053 val_acc=0.7812 val_macro_f1=0.7869
Epoch 055 | train_loss=0.0005 val_loss=0.0054 val_acc=0.7812 val_macro_f1=0.7869
Epoch 060 | train_loss=0.0007 val_loss=0.0049 val_acc=0.7812 val_macro_f1=0.7905
Epoch 065 | train_loss=0.000

,model_name,fold,best_epoch,best_selection_metric,best_selection_score,pretrained_loaded,freeze_mode,train_samples,val_samples,test_samples,...,total_params,checkpoint_path,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,test_accuracy,test_balanced_accuracy,test_macro_f1,test_weighted_f1
0,localization_no_pretrain,1,13,macro_f1,0.140000,False,full,130,33,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.212121,0.333333,0.140000,0.118788,0.238095,0.500000,0.168971,0.126692
1,localization_no_pretrain,2,25,macro_f1,0.247748,False,full,130,33,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.212121,0.500000,0.247748,0.079170,0.238095,0.500000,0.181159,0.091787
2,localization_no_pretrain,3,19,macro_f1,0.328947,False,full,130,33,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.212121,0.500000,0.328947,0.087719,0.190476,0.222222,0.081633,0.069971
3,localization_no_pretrain,4,8,macro_f1,0.119658,False,full,131,32,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.218750,0.333333,0.119658,0.078526,0.214286,0.250000,0.088235,0.075630
4,localization_no_pretrain,5,18,macro_f1,0.152174,False,full,131,32,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.218750,0.333333,0.152174,0.133152,0.238095,0.500000,0.157983,0.113005
5,localization_frozen_backbone,1,3,macro_f1,0.613520,True,frozen_backbone,130,33,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.818182,0.802198,0.613520,0.826898,0.785714,0.612847,0.602222,0.769524
6,localization_frozen_backbone,2,1,macro_f1,0.653097,True,frozen_backbone,130,33,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.848485,0.650641,0.653097,0.836285,0.785714,0.612847,0.602222,0.769524
7,localization_frozen_backbone,3,4,macro_f1,0.570882,True,frozen_backbone,130,33,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.727273,0.573718,0.570882,0.706734,0.785714,0.612847,0.602222,0.769524
8,localization_frozen_backbone,4,1,macro_f1,0.919485,True,frozen_backbone,131,32,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.906250,0.923077,0.919485,0.905344,0.785714,0.612847,0.602222,0.769524
9,localization_frozen_backbone,5,1,macro_f1,0.788304,True,frozen_backbone,131,32,42,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.781250,0.807692,0.788304,0.766009,0.785714,0.612847,0.602222,0.769524
